## 1. Imports

In [1]:
import requests
import pandas as pd
import time
import json
from pathlib import Path

## 2. Funções

In [15]:
# funçoes para coletar os dados 

LIMITE_TOTAL_EDITAIS = 3000

def coletarEditaisModalidade(codigo_modalidade, data_inicial, data_final, limite_restante):
    """Coleta editais de uma modalidade específica, paginando automaticamente,
    respeitando um limite máximo de registros a coletar."""
    editais = []
    pagina = 1
    
    while True:
        if len(editais) >= limite_restante:
            print(f"Limite de coleta atingido para esta modalidade. Parando.")
            break
        
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "codigoModalidadeContratacao": codigo_modalidade,
            "pagina": pagina,
            "tamanhoPagina": TAMANHO_PAGINA
        }
        
        response = requests.get(BASE_URL, params=params)
        
        if response.status_code == 429:
            print(f"Rate limit atingido na página {pagina}. Aguardando 5 segundos...")
            time.sleep(5)
            continue
        
        if response.status_code != 200:
            print(f"Erro na modalidade {codigo_modalidade}, página {pagina}: status {response.status_code}")
            break
        
        dados = response.json()
        registros = dados.get("data", [])
        
        if not registros:
            break
        
        editais.extend(registros)
        
        total_paginas = dados.get("totalPaginas", 1)
        print(f"Modalidade {codigo_modalidade} - página {pagina}/{total_paginas} - {len(registros)} registros (acumulado: {len(editais)})")
        
        if pagina >= total_paginas:
            break
        
        pagina += 1
        time.sleep(1.5)
    
    return editais


def coletar_todas_modalidades(modalidades, data_inicial, data_final, limite_total):
    """Roda a coleta para cada modalidade, parando quando o limite GLOBAL é atingido,
    independente de qual modalidade contribuiu mais — preserva o desbalanceamento real."""
    todos_editais = []
    
    for codigo, nome in modalidades.items():
        limite_restante = limite_total - len(todos_editais)
        
        if limite_restante <= 0:
            print(f"\nLimite total de {limite_total} já atingido. Pulando modalidade {nome}.")
            break
        
        print(f"\n--- Coletando: {nome} (código {codigo}) | restam {limite_restante} para o limite total ---")
        editais_modalidade = coletarEditaisModalidade(codigo, data_inicial, data_final, limite_restante)
        
        for edital in editais_modalidade:
            edital["modalidade_nome"] = nome
        
        todos_editais.extend(editais_modalidade)
        print(f"Total coletado

SyntaxError: unterminated string literal (detected at line 74) (514562803.py, line 74)

## 3. Coleta de dados - API do PNCP (Potal Nascional de Contas Publicas)

In [11]:
# configuraçoes essenciais para puxar os dados certos da API

BASE_URL = "https://pncp.gov.br/api/consulta/v1/contratacoes/publicacao"
DADOS_RAW_PATH = Path("../dados/raw")
DADOS_RAW_PATH.mkdir(parents=True, exist_ok=True)

DATA_INICIAL = "20260601"
DATA_FINAL = "20260603"

MODALIDADES = {
    4: "Concorrência - Eletrônica",
    5: "Concorrência - Presencial",
    6: "Pregão - Eletrônico",
    7: "Pregão - Presencial",
    8: "Dispensa de Licitação",
    9: "Inexigibilidade"
}

TAMANHO_PAGINA = 50